# 2. Datasets

Notebook 1 used one trial from `lund2013`. pEYES ships loaders for four public, human-annotated eye-tracking
datasets; each one downloads and caches on first use and returns a single, consistently-shaped `pandas.DataFrame`.
This notebook tours all four and loads a trial from a second dataset to show the pipeline is dataset-agnostic.

## The four built-in datasets

`peyes.datasets.get_metadata(name)` prints each dataset's source, license, and citation — check this before using
a dataset in your own work, since licenses differ.

In [1]:
import numpy as np
import peyes

for name in ["lund2013", "irf", "hfc", "gazecom"]:
    peyes.datasets.get_metadata(name)
    print()

Dataset Name:	Lund2013
URL:	https://github.com/richardandersson/EyeMovementDetectorEvaluation/archive/refs/heads/master.zip
License:	GNU GPL-3.0
Articles:
	Andersson, R., Larsson, L., Holmqvist, K., Stridh, M., & Nyström, M. (2017): One algorithm to rule them all? An evaluation and discussion of ten eye movement event-detection algorithms. Behavior Research Methods, 49(2), 616-637.

Dataset Name:	IRF
URL:	https://github.com/r-zemblys/irf/archive/refs/heads/master.zip
License:	MIT
Articles:
	Zemblys, Raimondas and Niehorster, Diederick C and Komogortsev, Oleg and Holmqvist, Kenneth. Using machine learning to detect events in eye-tracking data. Behavior Research Methods, 50(1), 160–181 (2018).

Dataset Name:	HFC
URL:	https://github.com/dcnieho/humanFixationClassification/archive/refs/heads/master.zip
License:	CC NC-BY-SA 4.0
Articles:
	Hooge, I.T.C., Niehorster, D.C., Nyström, M., Andersson, R. & Hessels, R.S. (2018). Is human classification by experienced untrained observers a gold stan

## Loading `lund2013`

Each loader (`lund2013`, `irf`, `hfc`, `gazecom`) takes a `directory` to cache to and returns the full dataset as
one DataFrame — every trial from every subject, stacked.

In [2]:
lund2013 = peyes.datasets.lund2013(directory="data", save=True, verbose=False)
lund2013.head()

,trial_id,subject_id,stimulus_type,stimulus_name,t,x,y,pupil,pixel_size,viewer_distance,MN,RA
0,1,TH20,moving_dot,1,0.0,123.2532,22.6264,NaN,0.037824,67.0,1.0,1.0
1,1,TH20,moving_dot,1,2.0,123.5395,22.9064,NaN,0.037824,67.0,1.0,1.0
2,1,TH20,moving_dot,1,4.0,123.2230,21.9909,NaN,0.037824,67.0,1.0,1.0
3,1,TH20,moving_dot,1,6.0,123.1883,21.7740,NaN,0.037824,67.0,1.0,1.0
4,1,TH20,moving_dot,1,8.0,125.0540,21.1805,NaN,0.037824,67.0,1.0,1.0


In [3]:
print(f"{lund2013['trial_id'].nunique()} trials, {lund2013['subject_id'].nunique()} subjects")
lund2013["stimulus_type"].value_counts()

63 trials, 30 subjects


stimulus_type
video         274096
image          87790
moving_dot     21326
Name: count, dtype: int64

Columns after the standard `t`/`x`/`y`/`pupil`/`pixel_size`/`viewer_distance` are human-rater label columns —
`lund2013` has two independent raters, `RA` and `MN` (not every trial has both; notebook 5 picks a trial rated by
both to compare raters directly).

## A trial from a different dataset: `irf`

`irf` (Zemblys et al., 2018) has a different rater column (`RZ`) and much longer trials (~80 seconds each). To keep
detection fast for this demo, we only use the first 2.5 seconds of one trial — the pipeline from notebook 1 is
identical regardless of which dataset the samples came from.

In [4]:
irf = peyes.datasets.irf(directory="data", save=True, verbose=False)
print(irf.columns.tolist())

trial = irf[irf["trial_id"] == 1].iloc[:2500]  # first ~2.5s
t, x, y = trial["t"].values, trial["x"].values, trial["y"].values
pixel_size, viewer_distance = trial["pixel_size"].values[0], trial["viewer_distance"].values[0]

detector = peyes.create_detector("engbert", missing_value=np.nan, min_event_duration=4, pad_blinks_time=0)
labels, _ = detector.detect(t=t, x=x, y=y, pixel_size_cm=pixel_size, viewer_distance_cm=viewer_distance)
events = peyes.create_events(
    labels=labels, t=t, x=x, y=y, pupil=trial["pupil"].values, pixel_size=pixel_size, viewer_distance=viewer_distance,
)
peyes.summarize_events(events)["event_type"].value_counts()

['trial_id', 'subject_id', 'stimulus_type', 't', 'x', 'y', 'pupil', 'pixel_size', 'viewer_distance', 'RZ']


event_type
FIXATION    10
SACCADE     10
BLINK        1
Name: count, dtype: int64

## `hfc` and `gazecom`

`hfc` (fixation classification in adults and infants) and `gazecom` (natural dynamic scenes) load and slice the
same way — `peyes.datasets.hfc(...)` / `peyes.datasets.gazecom(...)` — each with its own rater column(s) and
license. We won't re-run the full pipeline on them here, but everything in this guide applies unchanged.

## What's next

**[3 Parsing Custom Data & Configuration](./3%20Parsing%20Custom%20Data%20%26%20Configuration.ipynb)** — using your
own recordings, not just the built-in datasets.